In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/dupliqddataset/QuoraQuestions.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
total_data = pd.read_csv('/kaggle/input/dupliqddataset/QuoraQuestions.csv')

In [4]:
# Separate few datas
data = total_data.sample(90000, random_state=2)
data.sample(5)

,id,qid1,qid2,question1,question2,is_duplicate
119165,119165,193548,193549,What is the infant mortality rate for Canada?,What is the infant mortality rate for Norway?,0
62113,62113,108329,108330,How do you trace the location of a TextPlus nu...,Is it possible to track a phone number location?,0
165021,165021,256324,256325,How does PCPartPicker.com keep its Amazon affi...,Why the most famous actors are in their 40，50s...,0
204692,204692,307667,307668,How can we know someone is recording our phone...,Why doesn't an iPhone allow you to record phon...,0
389219,389219,521699,521700,What are some songs that are easy to sign in ASL?,What are the best ASL song performances?,0


In [5]:
# Need to handle these missing datas
# data.isnull().sum()

# Optional -> we have no q1 as null
# data[data['question1'].isnull()]
# 363362	493340	493341	NaN	My Chinese name is Haichao Yu. What English na...	0
# Delete thid data
# data = data.drop(363362)

# data[data['question2'].isnull()]
# 105780	174363	174364	How can I develop android app?	NaN	0
# 201841	303951	174364	How can I create an Android app?	NaN	0
# Two q2 are null. So we delete record with id 201841 and fill the q2 at the q2 with the id 105780 with the same.
# data = data.drop(201841)

# Update
# data.loc[data['id'] == 201841, 'question2'] = "How can I develop android app?"
# # Update the is_duplicate to 1
# data.loc[data['id'] == 201841, 'is_duplicate'] = 1
# data.loc[data['id'] == 201841]


In [6]:
print((data["is_duplicate"].value_counts()/data["is_duplicate"].count()) * 100)

is_duplicate
0    63.161111
1    36.838889
Name: count, dtype: float64


In [7]:
# Regex library
import re

# Preprocess
def data_preprocess(question):
    # Convert to lowercase and remove white spaces
    question = question.lower().strip()

    # Replace certain special characters with their string equivalents
    question = question.replace('%', 'percent')
    question = question.replace('$', 'dollar')
    question = question.replace('₹', 'rupee')
    question = question.replace('€', 'euro')
    question = question.replace('@', 'at')

    # The pattern '[math]' has no meaning so replace with empty string
    question = question.replace('[math]', '')

    # Replacing some numbers with string equivalents (not perfect, can be done better to account for more cases)
    question = question.replace('1,000,000,000', 'b')
    question = question.replace('1,000,000', 'm')
    question = question.replace('1,000', 'k')

    question = re.sub(r'([0-9]+)e{9}', r'\1b', question)
    question = re.sub(r'([0-9]+)e{6}', r'\1m', question)
    question = re.sub(r'([0-9]+)e{3}', r'\1k', question)

    contractions = { 
      "ain't": "am not / are not / is not / has not / have not",
      "aren't": "are not / am not",
      "can't": "cannot",
      "can't've": "cannot have",
      "'cause": "because",
      "could've": "could have",
      "couldn't": "could not",
      "couldn't've": "could not have",
      "didn't": "did not",
      "doesn't": "does not",
      "don't": "do not",
      "hadn't": "had not",
      "hadn't've": "had not have",
      "hasn't": "has not",
      "haven't": "have not",
      "he'd": "he had / he would",
      "he'd've": "he would have",
      "he'll": "he shall / he will",
      "he'll've": "he shall have / he will have",
      "he's": "he has / he is",
      "how'd": "how did",
      "how'd'y": "how do you",
      "how'll": "how will",
      "how's": "how has / how is / how does",
      "I'd": "I had / I would",
      "I'd've": "I would have",
      "I'll": "I shall / I will",
      "I'll've": "I shall have / I will have",
      "I'm": "I am",
      "I've": "I have",
      "isn't": "is not",
      "it'd": "it had / it would",
      "it'd've": "it would have",
      "it'll": "it shall / it will",
      "it'll've": "it shall have / it will have",
      "it's": "it has / it is",
      "let's": "let us",
      "ma'am": "madam",
      "mayn't": "may not",
      "might've": "might have",
      "mightn't": "might not",
      "mightn't've": "might not have",
      "must've": "must have",
      "mustn't": "must not",
      "mustn't've": "must not have",
      "needn't": "need not",
      "needn't've": "need not have",
      "o'clock": "of the clock",
      "oughtn't": "ought not",
      "oughtn't've": "ought not have",
      "shan't": "shall not",
      "sha'n't": "shall not",
      "shan't've": "shall not have",
      "she'd": "she had / she would",
      "she'd've": "she would have",
      "she'll": "she shall / she will",
      "she'll've": "she shall have / she will have",
      "she's": "she has / she is",
      "should've": "should have",
      "shouldn't": "should not",
      "shouldn't've": "should not have",
      "so've": "so have",
      "so's": "so as / so is",
      "that'd": "that would / that had",
      "that'd've": "that would have",
      "that's": "that has / that is",
      "there'd": "there had / there would",
      "there'd've": "there would have",
      "there's": "there has / there is",
      "they'd": "they had / they would",
      "they'd've": "they would have",
      "they'll": "they shall / they will",
      "they'll've": "they shall have / they will have",
      "they're": "they are",
      "they've": "they have",
      "to've": "to have",
      "wasn't": "was not",
      "we'd": "we had / we would",
      "we'd've": "we would have",
      "we'll": "we will",
      "we'll've": "we will have",
      "we're": "we are",
      "we've": "we have",
      "weren't": "were not",
      "what'll": "what shall / what will",
      "what'll've": "what shall have / what will have",
      "what're": "what are",
      "what's": "what has / what is",
      "what've": "what have",
      "when's": "when has / when is",
      "when've": "when have",
      "where'd": "where did",
      "where's": "where has / where is",
      "where've": "where have",
      "who'll": "who shall / who will",
      "who'll've": "who shall have / who will have",
      "who's": "who has / who is",
      "who've": "who have",
      "why's": "why has / why is",
      "why've": "why have",
      "will've": "will have",
      "won't": "will not",
      "won't've": "will not have",
      "would've": "would have",
      "wouldn't": "would not",
      "wouldn't've": "would not have",
      "y'all": "you all",
      "y'all'd": "you all would",
      "y'all'd've": "you all would have",
      "y'all're": "you all are",
      "y'all've": "you all have",
      "you'd": "you had / you would",
      "you'd've": "you would have",
      "you'll": "you shall / you will",
      "you'll've": "you shall have / you will have",
      "you're": "you are",
      "you've": "you have"
    }

    question_decontracted = []

    for word in question.split():
        if word in contractions:
            word = contractions[word]

        question_decontracted.append(word)

    question = ' '.join(question_decontracted)
    question = question.replace("'ve", " have")
    question = question.replace("n't", " not")
    question = question.replace("'re", " are")
    question = question.replace("'ll", " will")

    # Removing HTML tags
    question = re.sub(r'<.*?>', '', question)

    # Remove extra spaces
    question = re.sub(r'\s+', ' ', question)

    # Remove punctuations
    question = re.sub(r'[^\w\s]', '', question)
    
    return question

    

In [8]:
# Feature Engineering

# Adding new columns
# Questions length
data["q1_len"] = data["question1"].apply(lambda row: len(str(row)))
data["q2_len"] = data["question2"].apply(lambda row: len(str(row)))

# Words count
data["q1_words"] = data["question1"].apply(lambda row: len(str(row).split()))
data["q2_words"] = data["question2"].apply(lambda row: len(str(row).split()))

# Print the question where the question type is float
# data[data['question2'].apply(lambda x: isinstance(x, float))]
# 363362	493340	493341	NaN	My Chinese name is Haichao Yu. What English na...	0	3	123	1	21	NaN
# Some question1 and questions2 were null. We processed those questions

# Common words
def common_words(row):
    q1 = set(map(lambda word: word.lower().strip(), row["question1"].split(" ")))
    q2 = set(map(lambda word: word.lower().strip(), row["question2"].split(" ")))
    return len(q1 & q2)

# Optional checking
# data[data["q1_words"] == 1]
# Apply the function
data['word_common'] = data.apply(common_words, axis=1);

# Total words
def total_words(row):
    q1 = set(map(lambda word: word.lower().strip(), row["question1"].split(" ")))
    q2 = set(map(lambda word: word.lower().strip(), row["question2"].split(" ")))
    return (len(q1) + len(q2))

# Apply the function
data['word_total'] = data.apply(total_words, axis=1);

# Word share
data['word_share'] = round(data['word_common']/data['word_total'], 2)
data.sample(5)

AttributeError: 'float' object has no attribute 'split'

In [ ]:
# Do advanced feture engineering
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def fetch_token_features(row):

  q1 = row['question1']
  q2 = row['question2']

  # As some question length might be 0 so take safe margin divisor
  SAVE_DIV = 0.0001

  # STOP_WORDS = stopwords.words('english')

  # Define feature list
  token_features = [0.0]*8

  # Tokenize
  q1_tokens = q1.split()
  q2_tokens = q2.split()

  # If any question length is 0 then return the token feature list
  if len(q1_tokens) == 0 or len(q2_tokens) == 0:
    return token_features

  # Get non stop words
  q1_words = set([word for word in q1_tokens if word not in stop_words])
  q2_words = set([word for word in q2_tokens if word not in stop_words])

  # Get stop words
  q1_stops = set([word for word in q1_tokens if word in stop_words])
  q2_stops = set([word for word in q2_tokens if word in stop_words])

  # Common non stop words count
  common_word_count = len(q1_words.intersection(q2_words))

  # Common stop words count
  common_stop_count = len(q1_stops.intersection(q2_stops))

  # Common token count
  common_token_count = len(set(q1_tokens).intersection(set(q2_tokens)))

  token_features[0] = common_word_count / (min(len(q1_words),  len(q2_words)) + SAVE_DIV)
  token_features[1] = common_word_count / (max(len(q1_words),  len(q2_words)) + SAVE_DIV)
  token_features[2] = common_stop_count / (min(len(q1_stops),  len(q2_stops)) + SAVE_DIV)
  token_features[3] = common_stop_count / (max(len(q1_stops),  len(q2_stops)) + SAVE_DIV)
  token_features[4] = common_token_count / (min(len(q1_tokens),  len(q2_tokens)) + SAVE_DIV)
  token_features[5] = common_token_count / (max(len(q1_tokens),  len(q2_tokens)) + SAVE_DIV)

  # Similarity of last word
  token_features[6] = int(q1_tokens[-1] == q2_tokens[-1])

  # Similarity of first word
  token_features[7] = int(q1_tokens[0] == q2_tokens[0])

  return token_features
  

In [ ]:
# Apply fetch_token_features
token_features = data.apply(fetch_token_features, axis=1)

data["cwc_min"] = list(map(lambda x: x[0], token_features))
data["cwc_max"] = list(map(lambda x: x[1], token_features))
data["csc_min"] = list(map(lambda x: x[2], token_features))
data["csc_max"] = list(map(lambda x: x[3], token_features))
data["ctc_min"] = list(map(lambda x: x[4], token_features))
data["ctc_max"] = list(map(lambda x: x[5], token_features))
data["last_word_eq"] = list(map(lambda x: x[6], token_features))
data["first_word_eq"] = list(map(lambda x: x[7], token_features))

data.head(5)

In [ ]:
!pip install distance
import distance

# Fetch length features
def fetch_length_features(row):
  q1 = row['question1']
  q2 = row['question2']

  length_features = [0.0] * 3

  # Tokenize
  q1_tokens = q1.split()
  q2_tokens = q2.split()

  if len(q1_tokens) == 0 or len(q2_tokens) == 0:
    return length_features

  # Absolute length feature
  length_features[0] = abs(len(q1_tokens) - len(q2_tokens))

  # Average token length feature
  length_features[1] = (len(q1_tokens) + len(q2_tokens))/2
    
  longestSubstring = list(distance.lcsubstrings(q1, q2))

  if longestSubstring:
    length_features[2] = len(longestSubstring[0]) / (min(len(q1), len(q2)) + 1)
    
  else:
    length_features[2] = 0.0  # or some other default value
  

  return length_features

In [ ]:
# Apply fetch_length_features
# length_features = data.apply(fetch_length_features, axis=1)

data['abs_len_diff'] = list(map(lambda x: x[0], length_features))
data['mean_len'] = list(map(lambda x: x[1], length_features))
data['longest_substr_ratio'] = list(map(lambda x: x[2], length_features))

data.head(5)


In [ ]:
# Fuzzy features
!pip install fuzzywuzzy
from fuzzywuzzy import fuzz

def fetch_fuzzy_features(row):
  q1 = row['question1']
  q2 = row['question2']

  fuzzy_features = [0.0]*4

  # fuzz ratio
  fuzzy_features[0] = fuzz.QRatio(q1, q2)

  # fuzz partial ratio
  fuzzy_features[1] = fuzz.partial_ratio(q1, q2)

  # token sort ratio
  fuzzy_features[2] = fuzz.token_sort_ratio(q1, q2)

  # token set ratio
  fuzzy_features[3] = fuzz.token_set_ratio(q1, q2)

  return fuzzy_features

In [ ]:
# Apply fuzzy features
fuzzy_features = data.apply(fetch_fuzzy_features, axis=1)

data['fuzz_ratio'] = list(map(lambda x: x[0], fuzzy_features))
data['fuzz_partial_ratio'] = list(map(lambda x: x[1], fuzzy_features))
data['token_sort_ratio'] = list(map(lambda x: x[2], fuzzy_features))
data['token_set_ratio'] = list(map(lambda x: x[3], fuzzy_features))

data.head(2)
print(data.shape)

In [ ]:
# Retrieve questions only
questions = data[['question1', 'question2']]
questions.head()

# Retrive features only
final_data = data.drop(columns = ['id', 'qid1', 'qid2', 'question1', 'question2'])
final_data.shape

In [ ]:
# Do Bow
from sklearn.feature_extraction.text import CountVectorizer
questionsList = list(questions['question1']) + list(questions['question2'])
cv = CountVectorizer(max_features=3000)
q1_arr, q2_arr = np.vsplit(cv.fit_transform(questionsList).toarray(), 2)

In [ ]:
# Concat the data
temp_df1 = pd.DataFrame(q1_arr, index=questions.index)
temp_df2 = pd.DataFrame(q2_arr, index=questions.index)
temp_df = pd.concat([temp_df1, temp_df2], axis=1)
temp_df.shape

In [ ]:
final_data = pd.concat([final_data, temp_df], axis=1)
final_data.head()

In [ ]:
# Model training
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(final_data.iloc[:,1:].values,final_data.iloc[:,0].values,test_size=0.2,random_state=1)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier()
xgb.fit(X_train,y_train)
y_pred1 = xgb.predict(X_test)
accuracy_score(y_test,y_pred1)